# Aula 1, Self-attention

Notebook executável que acompanha a aula [01-self-attention.md](../../lessons/modulo-06-transformers/01-self-attention.md).

## O que vamos fazer aqui

A self-attention é a peça que substituiu a recorrência nos Transformers. Aqui vamos
implementá-la do zero e ler a matriz de atenção em uma frase curta. Para o resultado ficar
interpretável, construímos embeddings em que gato e felino são parecidos, e esperamos vê-los
atender mais um ao outro. Só numpy.

## A frase e os embeddings

Cada palavra é um vetor. Note que gato e felino têm vetores parecidos de propósito, e pulou
e alto são distintos de tudo.

In [ ]:
import numpy as np

def softmax(z, eixo=-1):
    z = z - z.max(axis=eixo, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=eixo, keepdims=True)


tokens = ["gato", "felino", "pulou", "alto"]
# Embeddings escolhidos: gato e felino são parecidos; pulou e alto, distintos.
E = np.array([
    [1.0, 0.0, 0.0, 0.0],   # gato
    [0.9, 0.1, 0.0, 0.0],   # felino (parecido com gato)
    [0.0, 0.0, 1.0, 0.0],   # pulou
    [0.0, 0.0, 0.0, 1.0],   # alto
])
print('embeddings:', E.shape)

## Calculando a self-attention

Sem projeções, para focar no mecanismo: as pontuações são os produtos escalares entre todos
os pares, escalados por raiz de d, e a softmax os transforma em pesos que somam 1 por linha.

In [ ]:
def self_attention(X):
    """Self-attention sem projeções, para focar no mecanismo."""
    d = X.shape[1]
    scores = X @ X.T / np.sqrt(d)       # alinhamento entre todos os pares
    A = softmax(scores, eixo=-1)        # pesos de atenção, somam 1 por linha
    return A @ X, A


saida, A = self_attention(E)
print("Matriz de atenção (linha = quem olha, coluna = para quem):")
for i, t in enumerate(tokens):
    print(f"  {t:7} {np.round(A[i], 2)}  soma = {A[i].sum():.2f}")

Cada linha soma 1, como esperado. E o padrão aparece: gato dá os maiores pesos a
gato e felino, e felino faz o mesmo, porque os embeddings são parecidos. Pulou e alto, sem
similaridade com os demais, concentram a atenção em si mesmos. A atenção é, no fundo, uma
média ponderada por similaridade, agora dentro da própria sequência. Para o projeto, faça um
visualizador que destaque, para cada token, qual outro recebe o maior peso.